# Final model persistence and production feature contract

Phase 13 packages the already-finalized Tuned XGBoost model for later inference. Selection, parameters, 42 ordered features, target and ten-minute horizon remain fixed. Run `.venv/Scripts/python.exe ml/src/save_final_model.py` to generate the canonical artifact. The script uses shared locked-model reconstruction, fits TRAIN + VALIDATION only, and requires reproduction of the authoritative Phase 11 held-out metrics before saving. TEST is used only for that equivalence gate, never fitting or new decisions.

The native XGBoost JSON artifact is `ml/models/final_xgboost_model.json`. It remains Git-ignored under existing policy; metadata lives in trackable `ml/results/final_model_metadata.json`. The script creates a temporary candidate, checks reload predictions, and requires any existing outputs to agree with deterministic regeneration before publishing. This notebook reads those outputs and uses the shared production helper; it does not refit or implement another persistence pipeline.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ml/src/save_final_model.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
assert Path(sys.executable).resolve() == (PROJECT_ROOT / '.venv/Scripts/python.exe').resolve()
from ml.src.model_inference import load_final_model, validate_feature_frame, predict_next_appliances, METADATA_PATH, ARTIFACT_PATH, read_json, sha256
from ml.src.save_final_model import RESULT_PATH, prediction_sha256
bundle = load_final_model()
model, metadata = bundle
record = read_json(RESULT_PATH)
print('Model:', metadata['model_name'])
print('Exact locked parameters:', metadata['locked_hyperparameters'])
print('Development rows:', metadata['development_rows'])
print('Development period:', metadata['development_start_timestamp'], 'through', metadata['development_end_timestamp'])
print('Native format:', metadata['model_format'])
print('Artifact:', metadata['model_artifact_relative_path'])
print('Artifact SHA-256:', sha256(ARTIFACT_PATH))
print('Software versions:', metadata['dependencies'], '| Python:', metadata['python_version'])
assert sha256(METADATA_PATH) == record['metadata_sha256']
assert sha256(ARTIFACT_PATH) == record['artifact_sha256']
assert metadata['test_used_for_fitting'] is False

Model: Tuned XGBoost
Exact locked parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 100, 'n_jobs': 1, 'objective': 'reg:squarederror', 'random_state': 42, 'subsample': 1.0, 'tree_method': 'hist', 'verbosity': 0}
Development rows: 15917
Development period: 2016-01-18 17:00:00 through 2016-05-08 05:40:00
Native format: XGBoost native JSON
Artifact: ml/models/final_xgboost_model.json
Artifact SHA-256: 6418bca4fd34b120928bdcc7c37c72e32f2cb6b8e5fd48d6164441aa1a9f7404
Software versions: {'numpy': '2.5.4', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'xgboost': '3.4.1'} | Python: 3.13.14


## Schema contract and inference

Inputs are already-engineered observations: exactly the 42 schema features in their saved order. The helper rejects missing, extra, duplicate or reordered columns, nonnumeric/complex/boolean inputs, missing values and infinities. It does not coerce, impute, construct lags or reorder arbitrary inputs. Current Appliances remains a feature; date and target do not.

Output is numeric predicted Appliances energy **Wh at t+10 minutes**, not kW. No clipping or confidence intervals are added. Load once and pass the returned `(model, metadata)` bundle to reuse it across calls. The following actual TRAIN sample is used for inference verification only.

In [2]:
sample = pd.read_csv(PROJECT_ROOT / 'data/processed/energy_train.csv', nrows=128, float_precision='round_trip').loc[:, metadata['ordered_feature_names']]
validate_feature_frame(sample, metadata['ordered_feature_names'])
forecasts = predict_next_appliances(sample, bundle)
display(forecasts.head().to_frame())
print('Output units/horizon:', forecasts.attrs)
assert prediction_sha256(forecasts.to_numpy()) == metadata['verification_sample']['predictions_float32_le_sha256']
print('Native serialization reload verification:', record['reload_verification'])
assert record['reload_verification']['exact_prediction_equality']
assert record['reload_verification']['max_absolute_prediction_difference_wh'] == 0.0
try:
    predict_next_appliances(sample.drop(columns='Appliances'), bundle)
except ValueError as error:
    print('Expected rejection:', error)
else:
    raise AssertionError('Missing feature was not rejected')

,predicted_appliances_next_wh
0,55.505409
1,55.505409
2,55.505409
3,55.505409
4,48.991730


Output units/horizon: {'unit': 'Wh', 'forecast_horizon_minutes': 10}
Native serialization reload verification: {'exact_prediction_equality': True, 'max_absolute_prediction_difference_wh': 0.0, 'passed': True, 'sample': 'First 128 chronological TRAIN rows; no TEST reload predictions', 'verification_rows': 128}
Expected rejection: Feature set mismatch: missing=['Appliances']; unexpected=[]


## Behavior reproduction and limitations

The following values are the original Phase 11 final test estimates and their reproduction check, not new model-performance results. This notebook does not load TEST. Model fitting always excludes TEST. Source checksums preserve the Phase 10-12 results and split data.

The exact generation environment is recorded, with no volatile timestamp. Deterministic regeneration is verified within this environment; other versions/platforms are not claimed equivalent. Native model format preserves model state, while the separate metadata preserves the training configuration and feature contract. Supported JSON serialization is used explicitly; UBJ is XGBoost's default, but no compatibility warning required a format change.

A schema-valid frame does not prove correct units, timing, sensor meaning or historical context. Callers must supply genuine features engineered with the approved origin-time methodology. One building and a January-May dataset limit generalization. No API, recommendation rules or frontend changes are included. Phase 13 ends here.

Reference: [XGBoost native Model IO](https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html).

In [3]:
print('Authoritative Phase 11 metrics:', metadata['held_out_test_metrics_from_phase11'])
print('Reproduction gate:', record['phase11_metric_reproduction'])
assert record['phase11_metric_reproduction']['passed']
assert all(sha256(PROJECT_ROOT / path) == expected for path, expected in record['input_sha256'].items())
for key in ['test_used_for_fitting', 'tuning_performed', 'model_selection_changed', 'feature_selection_performed']:
    assert record[key] is False
print('All Phase 13 notebook checks passed.')

Authoritative Phase 11 metrics: {'mae': 25.891124725341797, 'r2': 0.5820133686065674, 'rmse': 59.11701146812037}
Reproduction gate: {'absolute_differences': {'mae': 0.0, 'r2': 0.0, 'rmse': 0.0}, 'atol': 1e-09, 'expected': {'mae': 25.891124725341797, 'r2': 0.5820133686065674, 'rmse': 59.11701146812037}, 'passed': True, 'purpose': 'Equivalence check only; Phase 11 remains the final held-out evaluation', 'reconstructed': {'mae': 25.891124725341797, 'r2': 0.5820133686065674, 'rmse': 59.11701146812037}, 'rtol': 1e-09}
All Phase 13 notebook checks passed.
